# llm.py — Interactive Test

Testa `build_user_prompt`, `parse_references`, `get_llm_provider` e `answer_question`.

As células 1–5 funcionam sem chaves de API (usam um provider falso).
As células 6–7 fazem chamadas reais à API — defina as variáveis `OPENAI_KEY` / `ANTHROPIC_KEY` antes de rodá-las.

Certifique-se de ter instalado os requisitos:
```bash
pip install -r requirements.txt
```

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

In [ ]:
from core.llm import (
    LLMResponse,
    LLMError,
    OpenAILLMProvider,
    AnthropicLLMProvider,
    get_llm_provider,
    build_user_prompt,
    parse_references,
    answer_question,
    SYSTEM_PROMPT,
)
from core.chunking import Chunk

print("Import OK")

## Helper

In [ ]:
def make_chunks(texts: list[str], filename: str = "doc.pdf") -> list[Chunk]:
    return [
        Chunk(text=t, filename=filename, page_number=i + 1, chunk_index=i)
        for i, t in enumerate(texts)
    ]


def print_response(resp: LLMResponse) -> None:
    print(f"Provider   : {resp.provider_used}")
    print(f"Model      : {resp.model}")
    print(f"Tokens     : {resp.tokens_used}")
    print(f"Latency ms : {resp.latency_ms}")
    print(f"References : {resp.referenced_chunk_indices}")
    print(f"Answer     : {resp.answer}")

## 1 — build_user_prompt

In [ ]:
chunks = make_chunks([
    "O motor WEG W22 possui eficiência IE3.",
    "A tensão nominal é 220/380 V.",
    "O torque máximo é 150 Nm.",
])

prompt = build_user_prompt("Qual é a tensão nominal?", chunks)
print(prompt)

# Each chunk must appear numbered 1-based
assert "[1]" in prompt and "[2]" in prompt and "[3]" in prompt
assert "Qual é a tensão nominal?" in prompt
assert "source: doc.pdf, page 1" in prompt
print("\nOK — prompt contém todos os excerpts e a pergunta")

## 2 — parse_references

In [ ]:
# Normal case: citation marker at end
raw = "A tensão nominal é 220/380 V. [USED: 2, 3]"
answer, indices = parse_references(raw)
print(f"Answer  : {repr(answer)}")
print(f"Indices : {indices}")
assert answer == "A tensão nominal é 220/380 V."
assert indices == [1, 2]  # 0-based
print("OK — extração e conversão para 0-based")

print()

# No citation marker
raw_no_ref = "Não encontrei informação suficiente."
answer2, indices2 = parse_references(raw_no_ref)
assert answer2 == raw_no_ref
assert indices2 == []
print("OK — ausência de marcador retorna lista vazia")

print()

# Single reference
raw_single = "O motor possui IE3. [USED: 1]"
answer3, indices3 = parse_references(raw_single)
assert indices3 == [0]
print("OK — referência única")

## 3 — get_llm_provider

In [ ]:
# GPT model → OpenAI provider
p = get_llm_provider("gpt-4o-mini", openai_key="sk-fake")
assert isinstance(p, OpenAILLMProvider)
assert p.model == "gpt-4o-mini"
assert p.name == "openai"
print("OK — gpt-4o-mini → OpenAILLMProvider")

# Claude model → Anthropic provider
p2 = get_llm_provider("claude-sonnet-4-6", anthropic_key="sk-ant-fake")
assert isinstance(p2, AnthropicLLMProvider)
assert p2.model == "claude-sonnet-4-6"
assert p2.name == "anthropic"
print("OK — claude-sonnet-4-6 → AnthropicLLMProvider")

# Unsupported model → ValueError
try:
    get_llm_provider("llama-3", openai_key="sk-fake")
    assert False, "Deveria ter levantado ValueError"
except ValueError as e:
    print(f"OK — modelo não suportado levanta ValueError: {e}")

# Missing key → ValueError
try:
    get_llm_provider("gpt-4o-mini")  # no key
    assert False, "Deveria ter levantado ValueError"
except ValueError as e:
    print(f"OK — chave ausente levanta ValueError: {e}")

## 4 — answer_question com provider falso

In [ ]:
import asyncio


class FakeProvider:
    name = "fake"
    model = "fake-model"

    def __init__(self, text: str, tokens: int = 42):
        self._text = text
        self._tokens = tokens

    async def generate(self, system_prompt: str, user_prompt: str, max_tokens: int = 1000):
        return self._text, self._tokens


chunks = make_chunks([
    "O motor WEG W22 possui eficiência IE3.",
    "A tensão nominal é 220/380 V.",
])

fake = FakeProvider("A tensão nominal é 220/380 V. [USED: 2]")
resp = asyncio.run(answer_question("Qual é a tensão?", chunks, fake))

print_response(resp)

assert resp.answer == "A tensão nominal é 220/380 V."
assert resp.referenced_chunk_indices == [1]  # 0-based index of chunk 2
assert resp.tokens_used == 42
assert resp.provider_used == "fake"
assert resp.latency_ms >= 0
print("\nOK — answer_question retorna LLMResponse correto")

## 5 — answer_question: sem referências na resposta

In [ ]:
fake_no_ref = FakeProvider("Não encontrei informação suficiente nos documentos fornecidos.")
resp2 = asyncio.run(answer_question("Pergunta sem resposta?", chunks, fake_no_ref))

print_response(resp2)

assert resp2.referenced_chunk_indices == []
assert "Não encontrei" in resp2.answer
print("\nOK — lista de referências vazia quando LLM não cita excerpts")

## 6 — Chamada real: OpenAI

Defina `OPENAI_KEY` com sua chave antes de rodar esta célula.

In [ ]:
OPENAI_KEY = ""  # coloque sua chave aqui

if not OPENAI_KEY:
    print("OPENAI_KEY não definida — pulando teste real")
else:
    chunks = make_chunks([
        "O motor WEG W22 IE3 possui potência de 7,5 kW e tensão nominal de 220/380 V.",
        "A corrente nominal em 220 V é 28,8 A e em 380 V é 16,7 A.",
        "O fator de serviço é 1,15 e a classe de isolamento é F.",
    ])

    provider = get_llm_provider("gpt-4o-mini", openai_key=OPENAI_KEY)
    resp = asyncio.run(answer_question("Qual é a potência do motor?", chunks, provider))

    print_response(resp)
    assert resp.provider_used == "openai"
    assert len(resp.answer) > 0
    print("\nOK — chamada real OpenAI")

## 7 — Chamada real: Anthropic

Defina `ANTHROPIC_KEY` com sua chave antes de rodar esta célula.

In [ ]:
ANTHROPIC_KEY = ""  # coloque sua chave aqui

if not ANTHROPIC_KEY:
    print("ANTHROPIC_KEY não definida — pulando teste real")
else:
    chunks = make_chunks([
        "O motor WEG W22 IE3 possui potência de 7,5 kW e tensão nominal de 220/380 V.",
        "A corrente nominal em 220 V é 28,8 A e em 380 V é 16,7 A.",
        "O fator de serviço é 1,15 e a classe de isolamento é F.",
    ])

    provider = get_llm_provider("claude-haiku-4-5-20251001", anthropic_key=ANTHROPIC_KEY)
    resp = asyncio.run(answer_question("Qual é a corrente nominal em 380 V?", chunks, provider))

    print_response(resp)
    assert resp.provider_used == "anthropic"
    assert len(resp.answer) > 0
    print("\nOK — chamada real Anthropic")